In [1]:
import os
import torch
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import SegformerForSemanticSegmentation
from tqdm import tqdm
from pathlib import Path

In [2]:
class ValDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size):
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.mask_filenames = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
        
        self.image_path_map = {}
        for path in Path(image_dir).rglob("*.jpg"):
            self.image_path_map[path.name] = str(path)
            
    def __len__(self): return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_name = self.mask_filenames[idx]
        img_name = mask_name.replace('.png', '.jpg')
        img_path = self.image_path_map.get(img_name)
        
        if img_path is None:
            return torch.zeros((3, *self.img_size)), torch.zeros(self.img_size, dtype=torch.long), ""

        image = Image.open(img_path).convert("RGB").resize(self.img_size, Image.BILINEAR)
        mask = Image.open(os.path.join(self.mask_dir, mask_name)).convert("L").resize(self.img_size, Image.NEAREST)

        img_tensor = transforms.ToTensor()(image)
        img_tensor = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img_tensor)
        mask_tensor = torch.tensor(np.array(mask), dtype=torch.long)

        return img_tensor, mask_tensor, mask_name

In [22]:
VAL_IMAGE_DIR = r"D:\Study\HumanStudy\Dataset\Validation\01.원천데이터"
VAL_MASK_DIR = r"D:\Study\HumanStudy\Dataset\Validation_Masks"
PRED_SAVE_DIR = r"D:\Study\HumanStudy\Dataset\Validation_Predictions" 
WEIGHT_PATH = r"best_segformer_b0_crack_hybrid_c104.pth"
IMG_SIZE = (512, 512)
os.makedirs(PRED_SAVE_DIR, exist_ok=True)

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

val_dataset = ValDataset(VAL_IMAGE_DIR, VAL_MASK_DIR, IMG_SIZE)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0", num_labels=11, ignore_mismatched_sizes=True, use_safetensors=True
)
model.load_state_dict(torch.load(WEIGHT_PATH, map_location=device, weights_only=True))
model.to(device)
model.eval()

cuda


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(160, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  

In [26]:
num_classes = 11
total_inter = torch.zeros(num_classes).to(device)
total_union = torch.zeros(num_classes).to(device)
total_correct = 0
total_pixels = 0

with torch.no_grad():
    for images, masks, mask_names in tqdm(val_loader):
        if mask_names[0] == "": continue
            
        images, masks = images.to(device), masks.to(device)

        logits = model(pixel_values=images).logits
        logits = torch.nn.functional.interpolate(logits, size=IMG_SIZE, mode="bilinear", align_corners=False)
        preds = torch.argmax(logits, dim=1) 

        total_correct += (preds == masks).sum().item()
        total_pixels += torch.numel(masks)

        for c in range(num_classes):
            pred_inds = (preds == c)
            target_inds = (masks == c)
            total_inter[c] += (pred_inds & target_inds).sum()
            total_union[c] += (pred_inds | target_inds).sum()
            
        preds_np = preds.cpu().numpy().astype(np.uint8)
        
        for i in range(len(mask_names)):
            save_path = os.path.join(PRED_SAVE_DIR, mask_names[i])
            cv2.imwrite(save_path, preds_np[i])

100%|████████████████████████████████████████████████████████████████████████████| 6250/6250 [1:01:54<00:00,  1.68it/s]


In [27]:
pixel_accuracy = total_correct / (total_pixels + 1e-6)
ious = total_inter / (total_union + 1e-6)
miou = ious.mean().item()

class_names = ["배경(0)", "균열(1)", "누수(2)", "백태(3)", "박리(4)", "망상균열(5)", "박락(6)", "재료분리(7)", "철근노출(8)", "파손(9)", "들뜸(10)"]

print(f"Pixel Accuracy: {pixel_accuracy * 100:.2f}%")
print(f"Mean IoU: {miou * 100:.2f}%\n")

for c in range(num_classes):
    if total_union[c] > 0:
        print(f" - {class_names[c]:<12}: {ious[c].item() * 100:.2f}%")


Pixel Accuracy: 97.89%
Mean IoU: 8.90%

 - 배경(0)       : 97.89%
 - 균열(1)       : 0.00%
 - 누수(2)       : 0.00%
 - 백태(3)       : 0.00%
 - 박리(4)       : 0.00%
 - 망상균열(5)     : 0.00%
 - 박락(6)       : 0.00%
 - 재료분리(7)     : 0.00%
 - 철근노출(8)     : 0.00%
 - 파손(9)       : 0.00%
 - 들뜸(10)      : 0.00%
